# Chapter 6b — RAG over Research Papers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzafarooq/advanced-rag-from-scratch/blob/main/colab_original_notebooks/Chapter_6b.ipynb)

Companion code for the **second half of Chapter 6** of *Build an Advanced RAG Application (From Scratch)*.

Same retrieve → augment → generate pipeline as 6a, but a very different corpus: 500 research-paper abstracts from the [DBLP-v10 dataset on Kaggle](https://www.kaggle.com/datasets/nechbamohammed/research-papers-dataset). New things in this notebook:

- **Document chunking** — papers can exceed the embedder's context window.
- **Direct `transformers`** instead of `sentence-transformers` — useful when you want pooling control.
- **Metadata filtering** by year — show only post-2010 papers, etc.
- **Persistent Qdrant** — collection survives across notebook restarts.



# 1. Setup

You need three keys in `colab secrets`:

- `OPEN_ROUTER_API_KEY` — for generation
- `KAGGLE_USERNAME` and `KAGGLE_KEY` — for downloading the dataset

Get a Kaggle token at https://www.kaggle.com/settings → *Create New Token*.


In [ ]:
from IPython.display import display, HTML
def set_css(*args, **kwargs):
    display(HTML('''
    <style>
      pre {
          white-space: pre-wrap;  /* Enable word-wrapping in code/output blocks */
      }
    </style>
    '''))

get_ipython().events.register('pre_run_cell', set_css)

In [ ]:
#install packages
!pip install sentence_transformers openai
!pip install plotly
!pip install matplotlib
!pip install -Uqqq rich gradio
!pip install qdrant_client
!pip install transformers
!pip install openai

In [ ]:


# Import basic libraries
import numpy as np
import os, random
from pathlib import Path
from getpass import getpass
from rich.markdown import Markdown
import torch
import sys
import csv
csv.field_size_limit(sys.maxsize)

# Import necessary classes from the Hugging Face Transformers library
from transformers import AutoTokenizer, AutoModel

## 2. Download the dataset (Kaggle)

`kagglehub` reads `KAGGLE_USERNAME` / `KAGGLE_KEY` from the environment. The dataset is ~150MB.


In [ ]:
from google.colab import userdata
OPEN_ROUTER_API_KEY = userdata.get('OPEN_ROUTER_API_KEY')

from openai import OpenAI
open_router_client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=OPEN_ROUTER_API_KEY,
)


In [ ]:
import kagglehub

# Authenticate
kagglehub.login()

In [ ]:
import os
os.getcwd()

In [ ]:
import kagglehub
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("nechbamohammed/research-papers-dataset")

print("Path to dataset files:", path)

In [ ]:
#check path of downloaded dataset
os.chdir(path)


In [ ]:
!ls

In [ ]:
df = pd.read_csv("dblp-v10.csv")

In [ ]:
#view dataset
df.head()

In [ ]:
df.shape

In [ ]:
#preserve original dataset copy before updating it
df_original = df.copy()

In [ ]:
#view copy
df_original.head(15)

In [ ]:
# drop null values from dataset
df.dropna(subset=['abstract'], inplace=True)
df.head(15)

In [ ]:
# Keep a small subset and turn each paper into content plus metadata.

df = df_original[:500]  # Using only 25 rows for demonstration
df.dropna(subset=['abstract'], inplace=True)
#df = df.dropna(axis=1)  # Drop columns with null values

# Prepare data with metadata for traceability
data = []
for row_num, row in df.iterrows():
    #content = " ".join([f"{col}: {row[col]}" for col in df.columns])
    if row['abstract'] != 'NaN' :
      content = row['abstract']
      data.append({
          "page_content": content,
          "metadata": {
              "source": row["title"],
              "authors" : row["authors"],
              "year" : row["year"],
              "venue" : row["venue"],
              "paper_id" : row["id"]

          }
      })

    else:
      print(row['abstract'],row["id"])


# data

In [ ]:
print(f'You have {len(data)} document(s) in your data')

# 3. Prepare a small slice for the demo

Full DBLP is too big to embed in a notebook session. We take 500 papers, drop those without abstracts, and shape each row into a `{page_content, metadata}` dict.


In [ ]:
# Split each paper into chunks while preserving its metadata.

def simple_recursive_split(docs, chunk_size=4000, chunk_overlap=200, separators=None):
    # Extract the main text and its associated metadata
    text = docs["page_content"]
    metadata = docs["metadata"]

    # Set default separators if none are provided
    if separators is None:
        separators = ["\n\n", "\n", " ", ".", ",", "\uff0c", "\u3001", "\uff0e", "\u3002"]

    # Helper function to recursively split text based on the separators
    def split_with_separators(t):
        # If the text is already within the chunk size, return it directly
        if len(t) <= chunk_size:
            return [t]

        # Attempt splitting by each separator in order
        for sep in separators:
            if sep and sep in t:
                parts = t.split(sep)
                chunks = []
                current = ""

                # Build chunks without exceeding the maximum chunk size
                for part in parts:
                    part += sep  # Reattach the separator to preserve structure
                    if len(current + part) <= chunk_size:
                        current += part
                    else:
                        if current:
                            chunks.append(current.strip())
                        current = part  # Start a new chunk

                # Add the final leftover chunk
                if current:
                    chunks.append(current.strip())

                # Recursively re-split chunks that are still too large
                result = []
                for chunk in chunks:
                    if len(chunk) > chunk_size:
                        result.extend(split_with_separators(chunk))
                    else:
                        result.append(chunk)
                return result

        # Fallback: if no separators are effective, split the text by fixed character lengths
        return [t[i:i + chunk_size] for i in range(0, len(t), chunk_size)]

    # Split the original text
    splits = split_with_separators(text)

    # Add overlap between chunks to preserve context between adjacent segments
    overlapped = []
    for i, chunk in enumerate(splits):
        if i == 0:
            # First chunk, no overlap
            overlapped.append({
                "page_content": chunk,
                "metadata": metadata
            })
        else:
            # For subsequent chunks, add overlap from the end of the previous chunk
            overlap = splits[i - 1][-chunk_overlap:]
            overlapped.append({
                "page_content": f"{overlap} {chunk}",
                "metadata": metadata
            })

    return overlapped

# Apply the chunking function to each document in the dataset
# This flattens all chunks into a single list
texts = [chunk for doc in data for chunk in simple_recursive_split(doc, 1024, 50)]

In [ ]:
print (f'You now have {len(texts)} document(s) in your data')
print (f'There are {len(texts[1]["page_content"])} characters in your document')

In [ ]:

texts[0]

# 4. Chunk the documents

Most abstracts fit comfortably in one chunk, but the same code handles longer papers. See `chunking.py`.


In [ ]:
# Load the embedding model and define a helper for text embeddings.


# Set device to CUDA if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load embedding model from HuggingFace and move it to the selected device
text_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True).to(device)


# Function to generate embeddings from text, utilizing the selected device
def get_text_embeddings(text):
    # Move input tensors to the selected device
    inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad(): # Disable gradient calculation for inference
        outputs = text_model(**inputs)
    # Move embeddings back to CPU for further processing if needed
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu()
    return embeddings[0].detach().numpy()

# Example usage of the function
text = "This is a test sentence."

# Get the embedding vector for the input text
embeddings = get_text_embeddings(text)

# Optionally, get the length of the embedding (number of dimensions)
text_embeddings_size = len(embeddings)

# Print the first 5 values of the embedding vector for inspection
print(embeddings[:5])



In [ ]:
# Generate embeddings for all chunks
# This operation can be parallelized or batched for larger datasets for better performance
text_embeded = [get_text_embeddings(document["page_content"]) for document in texts]

In [ ]:
len(text_embeded)

# 5. Generate embeddings (transformers, mean-pooled)

We load the model with `transformers` directly so we can see exactly how the pooling works.


In [ ]:
# Start an in-memory Qdrant client for local retrieval testing.

from qdrant_client import QdrantClient, models

# Create a new Qdrant client instance using in-memory storage
# ":memory:" means the data will be stored temporarily in RAM (not saved to disk)
client = QdrantClient(":memory:")

# Display the size (number of dimensions) of the text embeddings we generated earlier
# This is important because Qdrant needs to know the exact size of each vector to create a collection
text_embeddings_size

In [ ]:
# Create or reset the Qdrant collection for the paper vectors.

try:
    # Define the name of the collection we want to manage in Qdrant.
    # it stores a group of vectors and their associated metadata.
    collection_name = "research_collection"

    # Check whether the collection already exists in Qdrant.
    # This avoids attempting to create a collection with a name that's already taken.
    if client.collection_exists(collection_name):
        # If the collection already exists, delete it to ensure we're starting fresh.
        # This is useful when we want to reset the state (e.g., during development or re-indexing).
        client.delete_collection(collection_name=collection_name)

        # Output a message confirming the collection was deleted successfully.
        print(f"Collection '{collection_name}' deleted successfully.")

    # Proceed to create a new collection regardless of whether it was previously deleted or not.
    # This ensures we always end up with a clean, newly-created collection.
    client.create_collection(
        collection_name=collection_name,  # The name of the new collection being created

        # This includes the dimensionality (size) and the distance metric used for similarity.
        vectors_config=models.VectorParams(
            size=text_embeddings_size,       # The number of dimensions in each vector.
                                             # Must match the output size of your embedding model.
            distance=models.Distance.COSINE  # The distance function used for comparing vectors.
                                             # COSINE is commonly used for text embeddings as it measures angular similarity.
        ),
    )

    # Print a confirmation that the collection was created successfully.
    print(f"Collection '{collection_name}' created successfully.")

except Exception as e:
    print(f"An error occurred while setting up the collection: {e}")

In [ ]:
# Upload the chunk embeddings and metadata into Qdrant.

from uuid import uuid4

import numpy as np

# Upload all our text embeddings to the "demo_collection" in Qdrant
client.upload_points(
    collection_name="research_collection",  # Target collection where we want to store our vectors

    # Create a list of PointStruct objects, one for each text chunk
    points=[
        models.PointStruct(
            id=str(uuid4()),  # Generate a unique ID for each point (as a string)

            # Convert the embedding to a NumPy array, which is the expected format
            vector=np.array(text_embeded[idx]),

            # Attach payload — additional information stored with each vector
            # This allows us to retrieve the original text and its metadata later
            payload={
                "metadata": doc["metadata"],         # Source and row info
                "content": doc["page_content"]       # The full text chunk
            }
        )
        for idx, doc in enumerate(texts)  # Loop through all texts and match them to their embeddings
    ]
)


So far we used in-memory Qdrant for a quick demo; now we switch to persistent storage in Google Drive so the index survives notebook restarts.

In [ ]:
#connect your drive to access files
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from qdrant_client import QdrantClient, models

import os
import shutil

# Define the local directory path where Qdrant should store its data
# This is where vector collections and other database files will be saved
# For Google Colab users, this may point to a mounted Google Drive location
qdrant_data_dir = '/content/drive/MyDrive/Semantic_Search/qdrant_data_research'

# ----------------------------------------
# Forcefully remove the directory if it exists
# ----------------------------------------

try:
    # Remove the entire directory and its contents recursively
    # This is useful when you want to reset the Qdrant database from scratch
    shutil.rmtree(qdrant_data_dir)
    print(f"Directory '{qdrant_data_dir}' removed successfully.")

except FileNotFoundError:
    # If the directory does not exist, no need to worry — just proceed
    print(f"Directory '{qdrant_data_dir}' not found. Proceeding to create it.")

except OSError as e:
    # If there's a permission issue or the directory is in use, catch and report it
    print(f"Error removing directory '{qdrant_data_dir}': {e}")

# ----------------------------------------
# Recreate the directory
# ----------------------------------------

# Create the directory (and any missing parent directories) if it doesn't exist
# After deletion, this ensures a clean, fresh directory is in place for Qdrant
os.makedirs(qdrant_data_dir, exist_ok=True)
print(f"Directory '{qdrant_data_dir}' created.")

# ----------------------------------------
# Initialize Qdrant Client
# ----------------------------------------

# Initialize the Qdrant client, telling it to use the newly created directory for local storage
# This setup will persist vector data across sessions (e.g., in Google Drive)
client = QdrantClient(path=qdrant_data_dir)
print("Qdrant client initialized with fresh storage directory.")


In [ ]:
try:

    collection_name = "research_collection"

    # Check whether the collection already exists in Qdrant.
    # This avoids attempting to create a collection with a name that's already taken.
    if client.collection_exists(collection_name):
        # If the collection already exists, delete it to ensure we're starting fresh.
        # This is useful when you want to reset the state (e.g., during development or re-indexing).
        client.delete_collection(collection_name=collection_name)

        # Output a message confirming the collection was deleted successfully.
        print(f"Collection '{collection_name}' deleted successfully.")


    client.create_collection(
        collection_name=collection_name,  # The name of the new collection being created

        vectors_config=models.VectorParams(
            size=text_embeddings_size,       # The number of dimensions in each vector.
                                             # Must match the output size of your embedding model.
            distance=models.Distance.COSINE  # The distance function used for comparing vectors.
                                             # COSINE is commonly used for text embeddings as it measures angular similarity.
        ),
    )

    # Print a confirmation that the collection was created successfully.
    print(f"Collection '{collection_name}' created successfully.")

except Exception as e:
    # If any error occurs during the process (e.g., connection issues, invalid parameters),
    # it will be caught here and the error message will be printed.
    print(f"An error occurred while setting up the collection: {e}")

In [ ]:
from uuid import uuid4

import numpy as np

# Upload all our text embeddings to the "demo_collection" in Qdrant
client.upload_points(
    collection_name="research_collection",  # Target collection where we want to store our vectors

    # Create a list of PointStruct objects, one for each text chunk
    points=[
        models.PointStruct(
            id=str(uuid4()),  # Generate a unique ID for each point (as a string)

            # Convert the embedding to a NumPy array, which is the expected format
            vector=np.array(text_embeded[idx]),

            # Attach payload — additional information stored with each vector
            # This allows us to retrieve the original text and its metadata later
            payload={
                "metadata": doc["metadata"],         # Source and row info
                "content": doc["page_content"]       # The full text chunk
            }
        )
        for idx, doc in enumerate(texts)  # Loop through all texts and match them to their embeddings
    ]
)

# 6. Index in Qdrant

Same helpers as 6a; different collection name.




In [ ]:
# Run a sample semantic search against the paper collection.

query = get_text_embeddings('an autoassociative neural network with dynamic synapses')

# Perform a similarity search in Qdrant using the query vector
# This finds the most relevant text chunks (based on vector similarity)
text_hits = client.query_points(
    collection_name="research_collection",  # The name of the collection where vectors were stored
    query=query,                         # The query vector — what we want to find similar results to
    limit=3,                             # Limit the number of results to 3 most relevant chunks
).points                                 # Extract only the list of matching points (each with vector + payload)


In [ ]:

text_hits = client.query_points(
    collection_name="research_collection",  # The name of the collection where vectors were stored
    query=query,                         # The query vector — what we want to find similar results to
    limit=3,                             # Limit the number of results to 3 most relevant chunks
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.year",
                match=models.MatchValue(value=2006), # Filter for documents with year 2020
            )
        ]
    )
).points                                 # Extract only the list of matching points (each with vector + payload)

In [ ]:
text_hits

# 7. Query


In [ ]:
OPEN_ROUTER_API_KEY = userdata.get('OPEN_ROUTER_API_KEY')

open_router_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",  # Set the API endpoint to OpenRouter (not OpenAI)
    api_key=OPEN_ROUTER_API_KEY               # Use your OpenRouter API key for authentication
)


In [ ]:
# Wrap Qdrant retrieval in a helper that returns clean source records.

def query_qdrant(query, qdrant_client, limit=5):
    query_em = get_text_embeddings(query)

    text_hits = qdrant_client.query_points(
        collection_name="research_collection",
        query=query_em,
        limit=limit
    ).points

    results = []
    for i, point in enumerate(text_hits):
        results.append({
            'source_id': i + 1,
            'content': point.payload['content'],
            'metadata': point.payload['metadata']
        })

    return results

### With a year filter

Qdrant supports server-side filters on payload fields — pass them at query time.


In [ ]:
# Generate a cited answer from the retrieved paper chunks.

from openai import OpenAI

open_router_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPEN_ROUTER_API_KEY
)

def generate_answer(query):
    prompt = f"""
    Based on the following query, generate a comprehensive answer.
    Include citations [1][2] and mention authors, paper titles, and venues.
    Explain concepts clearly.

    Query: "{query}"
    Context: "{query_qdrant(query, client)}"

    Return in Markdown format.
    """

    stream = open_router_client.chat.completions.create(
        model="qwen/qwen3-8b",
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )

    output_text = ""
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            content = chunk.choices[0].delta.content
            output_text += content
            print(content, end="")

    return output_text, query_qdrant(query, client)

In [ ]:
query = """an autoassociative neural network with dynamic synapses"""

In [ ]:
response, sources=generate_answer(query)

In [ ]:
sources

# 8. Full RAG over papers


## Optional: Reopen the saved Qdrant index from Google Drive

Up to this point, Section 7 showed the full RAG flow after building the index in the current notebook session. The next section is an optional follow-up workflow for a different situation: you already ran the indexing steps earlier and saved the Qdrant data in Google Drive.

That means you do not need to download the dataset, rebuild chunks, or generate embeddings again. Instead, you can mount Drive, reopen the saved Qdrant directory, and continue directly with retrieval and answer generation.


In [ ]:
from qdrant_client import QdrantClient

from qdrant_client.http.models import Distance, VectorParams

from google.colab import userdata

from openai import OpenAI

# Import HTML and display tools from IPython
# These allow you to inject custom HTML or CSS into the notebook
from IPython.display import HTML, display

# Import necessary classes from the Hugging Face Transformers library
# AutoTokenizer handles breaking text into tokens
# AutoModel loads the pre-trained model used to compute vector embeddings
from transformers import AutoTokenizer, AutoModel

# Import the openai

import openai

**1. Define the Qdrant client first to connect to the vector database.**

In [ ]:
# Close existing client if it exists
try:
    client.close()
except:
    pass

client = QdrantClient(path='/content/drive/MyDrive/Semantic_Search/qdrant_data_research')

In [ ]:
# Open or recover the persistent Qdrant client stored in Drive.

# Attempt to initialize the Qdrant client
try:
    # Initialize the Qdrant client and set its storage path
    # This stores and retrieves the vector database in the specified directory on disk
    client = QdrantClient(path='/content/drive/MyDrive/Semantic_Search/qdrant_data_research')

except RuntimeError as e:
    # Catch the specific error that occurs when the Qdrant client is already running with this path
    if "already accessed by another instance" in str(e):
        print("Qdrant is already initialized with this path in the current session.")
        print("You don't need to create the client again — reuse the existing one.")
    else:
        # Re-raise the error if it's something else
        raise


**2. Define the OpenRouter client to serve as the language model (LLM) for the pipeline.**


In [ ]:
OPEN_ROUTER_API_KEY = userdata.get('OPEN_ROUTER_API_KEY')


open_router_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",  # Set the API endpoint to OpenRouter (not OpenAI)
    api_key=OPEN_ROUTER_API_KEY               # Use your OpenRouter API key for authentication
)


**3. Import the same embedding model used during vector database creation to ensure consistency.**

In [ ]:
# Load the same embedding model so query vectors stay compatible.


text_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

# Define a function to convert input text into a fixed-size vector (embedding)
def get_text_embeddings(text):
    # Tokenize the input text and return it as PyTorch tensors
    # padding=True: pad shorter sequences to ensure consistent length
    # truncation=True: cut off text that is too long for the model
    inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    # Pass the tokenized input through the model to obtain output embeddings
    outputs = text_model(**inputs)

    # outputs.last_hidden_state contains embeddings for each token
    # We take the mean across all tokens to get a single vector for the entire text
    embeddings = outputs.last_hidden_state.mean(dim=1)

    # Convert the result to a NumPy array and remove it from the computation graph
    return embeddings[0].detach().numpy()


**4. Test the retrieval functions to ensure they're returning relevant results.**

In [ ]:
query = """an autoassociative neural network with dynamic synapses"""

In [ ]:
# get query embedded
query_em = get_text_embeddings(query)

In [ ]:
text_hits = client.query_points(
        collection_name="research_collection",
        query=query_em,
        limit=10,
    ).points



In [ ]:
# Extract the original text content from each result returned by the similarity search
# `text_hits` is a list of points returned by Qdrant's query
# Each point has a `payload`, which contains metadata and the original text chunk

contents = [point.payload['content'] for point in text_hits]

In [ ]:
contents

In [ ]:
# Extract the metadata for each point returned by the similarity search
# Each result (point) has a payload dictionary that includes metadata stored when uploading the vectors

meta = [point.payload['metadata'] for point in text_hits]

In [ ]:
meta

In [ ]:
# Loop through each text chunk in the `contents` list
# These are the top-matching results returned by the Qdrant similarity search
for i in contents:
    # Print the actual text content
    print(i)

    # Print a separator line to clearly distinguish between different chunks
    print('###########')


**5. Create a retriever function to extract relevant chunks from the documents.**

In [ ]:
# Define a function to search the Qdrant vector database using a natural language query
def query_qdrant(query, qdrant_client, limit=5):
    # Step 1: Convert the query text into an embedding (vector representation)
    # This embedding will be compared with stored vectors in the collection
    query_em = get_text_embeddings(query)

    # Step 2: Query the Qdrant collection using the embedding
    # This finds the top `limit` most similar text chunks based on vector similarity
    text_hits = qdrant_client.query_points(
        collection_name="research_collection",  # The name of the Qdrant collection to search
        query=query_em,                     # The embedding of the input query
        limit=limit                         # Number of top results to return
    ).points                                 # Extract the matching points (results)

    # Step 3: Prepare the results in a clean format (text + metadata)
    results = []
    i = 1
    for point in text_hits:
        results.append({
            'source_id': i,
            'content': point.payload['content'],    # The original text content
            'metadata': point.payload['metadata']   # Associated metadata (e.g., title, row number)
        })

        i += 1

    # Return the list of results
    return results


In [ ]:
query_qdrant(query, client)

**6. Now, let's integrate everything by combining our Retrieval functiom with the Language Model to complete our RAG (Retrieval-Augmented Generation) pipeline.**

In [ ]:
# Define a function that uses a language model to generate an answer based on a user's query
def generate_answer(query):
    # Build the prompt that will be sent to the LLM
    # The prompt includes:
    # - Instructions to clean and format the answer
    # - The user's original query
    # - The context retrieved from Qdrant (via semantic search)
    prompt = f"""
    Based on the following query from a user, please generate a small answer
    focusing on the original query and the response given. The answer should be paragraphs.
    Remove the special characters and (/n), make the output clean and long.
    Please cite source for each part as [1][2].
    Mention Authors, paper title and venue.
    Just start with the answer, no need to give any salutations. Please explain the concept like I am 5.

    ###########
    query:
    "{query}"

    ########

    context:
    "{query_qdrant(query, client)}"
    #####

    Return in Markdown format.
    """

    # Send the prompt to the LLM using streaming mode
    # This allows the response to be received in real-time, piece by piece
    stream = open_router_client.chat.completions.create(
        model="qwen/qwen3-8b",  # Model to use (can be any OpenAI-compatible model)
        messages=[
            {
                "role": "user",
                "content": prompt,
            },
        ],
        stream=True,  # Enable streaming so we get partial output as it generates
    )

    # Initialize a variable to hold the full response
    output_text = ""

    # Iterate through the streaming response chunks
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            content = chunk.choices[0].delta.content
            output_text += content  # Append new content to the full output
            print(content, end="")  # Print each chunk live as it's received

    # Return both the final answer and the context used (for reference or display)
    return output_text, query_qdrant(query, client)


In [ ]:
response,sources = generate_answer(query)

In [ ]:
import markdown
from IPython.display import display, HTML

def render_markdown(md_text):
    # Convert Markdown to HTML
    html = markdown.markdown(md_text)
    # Display the HTML
    display(HTML(html))

In [ ]:
#for markdown layout
render_markdown(response)

In [ ]:
sources

## Time to Build a functional Gradio interface to interact with the RAG system.

In [ ]:
import gradio as gr

**1. Redefine our RAG function**

In [ ]:
import openai

# Define a function to generate a streamed answer to a user's query using an LLM
# This version includes error handling and uses Python's `yield` to stream results back as they're generated
def generate_answer(query):
    # Step 1: Try to get relevant context from Qdrant (vector search)
    try:
        sources = query_qdrant(query, client)
    except Exception as e:
        # If something goes wrong (e.g., Qdrant is not running), return a fallback message
        sources = [{"error": f"Error retrieving sources: {str(e)}"}]

    # Step 2: Prepare the prompt for the language model
    # Includes the user's question and the context retrieved from the vector database
    prompt = f"""
    Based on the following query from a user, please generate a small answer
    focusing on the original query and the response given. The answer should be paragraphs.
    Remove special characters and (/n); make the output clean and long.
    Please cite source for each part as [1][2]. Just start with the answer — no salutations.

    ###########
    query:
    "{query}"

    ########

    context:
    "{sources}"
    #####

    Return in Markdown format.
    """

    # Step 3: Send the prompt to the OpenRouter-compatible LLM (Qwen model)
    stream = open_router_client.chat.completions.create(
        model="qwen/qwen3-8b",
        messages=[
            {
                "role": "user",
                "content": prompt,
            },
        ],
        stream=True,  # Enable streaming response
    )

    # Step 4: Stream and yield the generated content chunk by chunk
    full_response = ""
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            content = chunk.choices[0].delta.content
            full_response += content

            # Yield lets us return partial results as they're received (for real-time feedback)
            yield full_response


**2. Create a Demo Interface**

In [ ]:
# Define example inputs for the UI — users can click these to try predefined queries
examples = [
    ["Democrats in Senate"],
    ["Climate Change Challenges in Europe"],
    ["Philosophy in the world of Minimalism"],
    ["Hate Speech vs Freedom of Speech"],
    ["Articles by Noam Chomsky on US Politics"],
    ["The importance of values and reflection"]
]

# Set up the Gradio interface
# - fn: the function to call when user enters input (must be a generator if using yield)
# - title: the name shown at the top of the web app
# - inputs: defines the input component (in this case, a text box)
# - outputs: defines what kind of output to display (Textbox with 3 lines labeled "Response")
# - examples: preloaded example queries for users to click and run

import gradio as gr

demo = gr.Interface(
    fn=generate_answer,  # The function that will process user input
    title="The Truth Serum",  # Title for the web app
    inputs="text",  # Single text input from the user
    outputs=gr.components.Textbox(lines=3, label="Response"),  # Output display
    examples=examples,  # List of sample queries for users to try
    live=False,  # Optional: set to True if you want real-time feedback as user types
)

# Launch the interface
# - share=True gives you a public link (useful in Colab or for sharing with others)
# - debug=True enables logging for error tracking
demo.launch(share=True, debug=True)


**3. Create a Demo Interface with Sources**

In [ ]:
import json

def sanitize_sources(sources):
    """Convert sources to a JSON-serializable format."""
    clean = []
    for source in sources:
        if isinstance(source, dict):
            # Recursively ensure all values are serializable
            clean.append({
                k: (v if isinstance(v, (str, int, float, bool, list, dict, type(None))) else str(v))
                for k, v in source.items()
            })
        else:
            # Handle Qdrant ScoredPoint or other custom objects
            try:
                # Qdrant objects often have a .payload and .score attribute
                clean.append({
                    "content": getattr(source, "payload", str(source)),
                    "score":   float(getattr(source, "score", 0)),
                    "id":      str(getattr(source, "id", ""))
                })
            except Exception:
                clean.append({"raw": str(source)})
    return clean


def generate_answer(query):
    try:
        sources = query_qdrant(query, client)
    except Exception as e:
        sources = [{"error": f"Error retrieving sources: {str(e)}"}]

    #  Sanitize before serializing
    clean_sources = sanitize_sources(sources)

    prompt = f"""
    Based on the following query from a user, please generate a small answer
    focusing on the original query and the response given. The answer should be paragraphs
    remove the special characters and (/n ), make the output clean and long. Please cite source for each part as [1][2]
    Just start with the answer, no need to give any salutations

    ###########
    query:
    "{query}"

    ########

    context:
    "{clean_sources}"
    #####

    Return in Markdown format.
    """

    stream = open_router_client.chat.completions.create(
        model="qwen/qwen3-8b",
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )

    # Serialize once, outside the loop — no repeated serialization
    sources_json = clean_sources  # gr.JSON accepts a dict/list directly

    full_response = ""
    for chunk in stream:
        if chunk.choices[0].delta.content is not None:
            full_response += chunk.choices[0].delta.content
            yield full_response, sources_json

    if not full_response:
        yield "No response generated", sources_json

examples = [
    ["Democrats in Senate"],
    ["Climate Change Challenges in Europe"],
    ["Philosophy in the world of Minimalism"],
    ["Hate Speech vs Freedom of Speech"],
    ["Articles by Noam Chomsky on US Politics"],
    ["The importance of values and reflection"]
]

demo = gr.Interface(
    fn=generate_answer,
    title="The Truth Serum",
    inputs="text",
    outputs=[
        gr.components.Textbox(lines=8, label="Response"),
        gr.components.JSON(label="Sources")
    ],
    examples=examples
)

demo.queue()
demo.launch(share=True, debug=True)

